# 🎙️ Audio Sentiment Analyzer — Qwen2.5-Omni-7B (T4 GPU / 4-bit Quantized)

## ⚙️ STEP 1: Install Dependencies


In [ ]:
# ======================== MAINQWEN.ipynb ========================
# MAINQWEN.ipynb -> Environment setup, dependency installation, & GPU validation
# ||
# ||
# ||
# imports -> Logic Flow -> Environment and hardware validation:
# ||                 |
# ||                 |--- Install dependencies -> Fetch pip/apt-get packages
# ||                 |--- Check versions -> Verify torch & transformers
# ||                 |--- IF torch.cuda.is_available()
# ||                 |    └── Verify GPU -> Check name & VRAM capacity
# ||                 |--- Try import bitsandbytes -> Check 4-bit readiness
# ||                 |--- Try import Qwen2_5Omni -> Check model support
# ||                 |--- Try import qwen_omni_utils -> Check util support
# ||                 |--- assert torch.cuda.is_available() -> Final GPU block
# ||
# ======================================================================

# ---------------------------------------------------------------
# SECTION: IMPORTS / CONSTANTS
# ---------------------------------------------------------------
!pip install -q -U 'transformers==4.52.0' accelerate
!pip install -q bitsandbytes>=0.43.0
!pip install -q librosa soundfile pydub noisereduce resampy soxr
!pip install -q qwen-omni-utils[decord]
!apt-get install -q -y ffmpeg

import importlib, sys, torch, transformers

# ---------------------------------------------------------------
# SECTION: MAIN / EXECUTION ENTRY POINT
# ---------------------------------------------------------------
print('━'*55)
print('✅ Dependency check:')
print(f'   torch version       : {torch.__version__}')
print(f'   transformers version: {transformers.__version__}')
print(f'   CUDA available      : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'   GPU                 : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM                : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

try:
    import bitsandbytes as bnb
    print(f'   bitsandbytes version: {bnb.__version__}')
    print('✅ bitsandbytes (4-bit quantization) found!')
except ImportError as e:
    print(f'❌ bitsandbytes not found: {e}')
    print('   Fix: !pip install bitsandbytes>=0.43.0, then restart runtime')

try:
    from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
    print('✅ Qwen2_5OmniForConditionalGeneration found!')
except ImportError as e:
    print(f'❌ Import failed: {e}')
    print('   Fix: re-run this cell, then Runtime → Restart and run all')

try:
    from qwen_omni_utils import process_mm_info
    print('✅ qwen_omni_utils.process_mm_info found!')
except ImportError as e:
    print(f'❌ qwen_omni_utils not found: {e}')

print('━'*55)
assert torch.cuda.is_available(), '❌ No GPU — go to Runtime → Change runtime type → T4 GPU'
print('✅ All checks passed! Proceed to Step 2.')

Reason for being yanked: <none given>
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 122.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 97.3 MB/s eta 0:00:00


## 📦 STEP 2: Imports & Configuration



In [ ]:
# ========================  ========================
# Setup config -> load env vars, & verify GPU
# ||
# ||
# ||
# Functions/Methods -> Logic Flow -> Init & Validation:
# ||                 |
# ||                 |--- load_dotenv() -> Load env vars
# ||                 |--- os.getenv() -> Get diarizer URL
# ||                 |--- os.makedirs() -> Create output dir
# ||                 |--- IF torch.cuda.is_available()
# ||                 |    └── torch.cuda.get_device_name() -> Fetch GPU name
# ||                 |    └── torch.cuda.get_device_properties() -> Calc VRAM
# ||
# ======================================================================

# ---------------------------------------------------------------
# SECTION: IMPORTS
# ---------------------------------------------------------------
import os, re, json, time, warnings, tempfile
from pathlib import Path
from datetime import datetime
import numpy as np
import librosa
import soundfile as sf
import noisereduce as nr
from pydub import AudioSegment
import torch
from qwen_omni_utils import process_mm_info

# ---------------------------------------------------------------
# SECTION: MAIN / EXECUTION ENTRY POINT
# ---------------------------------------------------------------
warnings.filterwarnings('ignore')

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# ---------------------------------------------------------------
# SECTION: CONSTANTS
# ---------------------------------------------------------------
CONFIG = {
    "model_id"                 : "Qwen/Qwen2.5-Omni-7B",
    "load_in_4bit"             : True,
    "bnb_4bit_compute_dtype"   : torch.bfloat16,
    "bnb_4bit_use_double_quant": True,
    "bnb_4bit_quant_type"      : "nf4",
    "target_sample_rate"       : 16000,
    "max_audio_duration"       : 25,
    "segment_overlap"          : 2,
    "noise_reduction_strength" : 0.5,
    "diarizer_url"             : os.getenv("DIARIZER_URL", "https://cristobal-lawless-colorationally.ngrok-free.dev/diarize"),
    "output_dir"               : "/content/sentiment_results",
    "save_cleaned_audio"       : True,
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)

print(f'✅ Config loaded.')
print(f'   Model      : {CONFIG["model_id"]}')
print(f'   Quantized  : 4-bit NF4 (bfloat16 compute)')
print(f'   Bridge URL : {CONFIG["diarizer_url"]}')
print(f'   Output dir : {CONFIG["output_dir"]}')

if torch.cuda.is_available():
    print(f'   GPU        : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'   Est. usage : ~5-6 GB (7B @ 4-bit) + ~1-2 GB inference overhead')
else:
    print(f'   GPU        : ❌ NOT FOUND')

In [5]:
"""
==============================================================================================
FLOW
  requests.get (requests) | Ping bridge health endpoint
    |
  r.json | Display bridge status
==============================================================================================
"""

import requests
r = requests.get("https://cristobal-lawless-colorationally.ngrok-free.dev/health")
print(r.json())

{'status': 'ok', 'pipeline_loaded': True, 'device': 'cpu'}


## 🎛️ STEP 3: Audio Preprocessing Pipeline


In [6]:
# ======================== AudioPreprocessor ========================
# AudioPreprocessor -> Format, resample, & segment audio
# ||
# ||
# ||
# Functions/Methods -> preprocess() -> Main audio pipeline
# ||                 |
# ||                 |---> __init__() -> Init config vars
# ||                 |---> convert_to_wav() -> Input -> .wav
# ||                 |---> load_and_resample() -> Load mono @ target Hz
# ||                 |---> reduce_noise() -> Bypassed for accuracy
# ||                 |---> normalize_audio() -> Bypassed for accuracy
# ||                 |---> trim_silence() -> Bypassed for duration
# ||                 |---> segment_audio() -> Split to overlapping chunks
# ||                 |
# ||                 |---> Logic Flow -> Pipeline execution:
# ||                                  |
# ||                                  |--- IF valid audio
# ||                                  |    └── preprocess() -> Format, load, segment
# ||                                  |--- IF ext != .wav
# ||                                  |    └── convert_to_wav() -> Return temp .wav
# ||                                  |--- IF duration > max
# ||                                  |    └── segment_audio() -> Return overlapping chunks
# ||
# ======================================================================

# ---------------------------------------------------------------
# SECTION: CLASSES
# ---------------------------------------------------------------
class AudioPreprocessor:

    def __init__(self, config: dict):
        self.sr             = config["target_sample_rate"]
        self.max_duration   = config["max_audio_duration"]
        self.overlap        = config["segment_overlap"]
        self.noise_strength = config["noise_reduction_strength"]
        self.save_cleaned   = config["save_cleaned_audio"]
        self.output_dir     = config["output_dir"]

    def convert_to_wav(self, audio_path: str) -> str:
        audio_path = Path(audio_path)
        suffix = audio_path.suffix.lower()
        if suffix == '.wav':
            return str(audio_path)
        print(f'  🔄 Converting {suffix} → .wav')
        audio = AudioSegment.from_file(str(audio_path))
        tmp_wav = tempfile.mktemp(suffix='.wav')
        audio.export(tmp_wav, format='wav')
        return tmp_wav

    def load_and_resample(self, wav_path: str) -> "np.ndarray":
        audio, _ = librosa.load(wav_path, sr=self.sr, mono=True, res_type='soxr_hq')
        print(f'  📊 Loaded: {len(audio)/self.sr:.1f}s | {self.sr}Hz | mono')
        return audio

    def reduce_noise(self, audio: "np.ndarray") -> "np.ndarray":
        print(f'  🔇 Noise reduction SKIPPED (disabled to preserve original duration)')
        return audio

    def normalize_audio(self, audio: "np.ndarray") -> "np.ndarray":
        print(f'  🔊 Normalization SKIPPED (disabled to preserve original duration)')
        return audio

    def trim_silence(self, audio: "np.ndarray") -> "np.ndarray":
        print(f'  ✂️  Silence trim SKIPPED (disabled to preserve original duration)')
        return audio

    def segment_audio(self, audio: "np.ndarray") -> list:
        total_duration = len(audio) / self.sr
        segments = []

        if total_duration <= self.max_duration:
            segments.append((0.0, total_duration, audio))
            print(f'  📍 Single segment: {total_duration:.1f}s')
        else:
            max_samples     = int(self.max_duration * self.sr)
            overlap_samples = int(self.overlap * self.sr)
            step            = max_samples - overlap_samples
            start, seg_num  = 0, 1
            while start < len(audio):
                end       = min(start + max_samples, len(audio))
                start_sec = start / self.sr
                end_sec   = end   / self.sr
                segments.append((start_sec, end_sec, audio[start:end]))
                print(f'  📍 Segment {seg_num}: {start_sec:.1f}s – {end_sec:.1f}s')
                start += step
                seg_num += 1
                if end == len(audio):
                    break
        return segments

    def preprocess(self, audio_path: str) -> dict:
        print(f'\n🎛️  PREPROCESSING: {Path(audio_path).name}')
        print('─' * 50)

        wav_path          = self.convert_to_wav(audio_path)
        audio             = self.load_and_resample(wav_path)
        original_duration = len(audio) / self.sr
        audio             = self.reduce_noise(audio)
        audio             = self.normalize_audio(audio)
        audio             = self.trim_silence(audio)
        segments          = self.segment_audio(audio)

        cleaned_path = None
        if self.save_cleaned:
            stem         = Path(audio_path).stem
            cleaned_path = os.path.join(self.output_dir, f'{stem}_cleaned.wav')
            sf.write(cleaned_path, audio, self.sr)
            print(f'  💾 Cleaned audio saved: {cleaned_path}')

        print(f'  ✅ Done: {len(segments)} segment(s)')
        return {
            'original_file'        : str(audio_path),
            'cleaned_audio_path'   : cleaned_path,
            'original_duration_sec': round(original_duration, 2),
            'final_duration_sec'   : round(len(audio) / self.sr, 2),
            'sample_rate'          : self.sr,
            'num_segments'         : len(segments),
            'segments'             : segments,
        }

# ---------------------------------------------------------------
# SECTION: MAIN / EXECUTION ENTRY POINT
# ---------------------------------------------------------------
print('✅ AudioPreprocessor defined.')

✅ AudioPreprocessor defined.


## 🤖 STEP 4: Load Qwen2.5-Omni-7B Model (4-bit Quantized)


In [ ]:
# ======================== Model & Processor Initialization ========================
# Model Initialization -> Loads 4-bit quantized Qwen Omni model and multimodal processor.
# ||
# ||
# ||
# Functions/Methods -> gc.collect() -> Free Python memory
# ||                 |
# ||                 |---> torch.cuda.empty_cache() -> Clear unused VRAM
# ||                 |---> BitsAndBytesConfig() -> Define 4-bit settings
# ||                 |---> from_pretrained() -> Load multimodal processor & model weights
# ||                 |---> eval() -> Set inference mode
# ||                 |---> torch.cuda.memory_allocated() -> Check VRAM used
# ||                 |---> torch.cuda.get_device_properties() -> Verify VRAM state
# ||
# ======================================================================

# ---------------------------------------------------------------
# SECTION: IMPORTS
# ---------------------------------------------------------------
from transformers import (
    Qwen2_5OmniForConditionalGeneration,
    Qwen2_5OmniProcessor,
    BitsAndBytesConfig,
)
import gc

# ---------------------------------------------------------------
# SECTION: MAIN / EXECUTION ENTRY POINT
# ---------------------------------------------------------------
gc.collect()
torch.cuda.empty_cache()
print(f'GPU before load: {torch.cuda.memory_allocated()/1e9:.2f} GB used')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=CONFIG['load_in_4bit'],
    bnb_4bit_compute_dtype=CONFIG['bnb_4bit_compute_dtype'],
    bnb_4bit_use_double_quant=CONFIG['bnb_4bit_use_double_quant'],
    bnb_4bit_quant_type=CONFIG['bnb_4bit_quant_type'],
)

print(f'\n⏳ Loading processor: {CONFIG["model_id"]}')
processor = Qwen2_5OmniProcessor.from_pretrained(CONFIG['model_id'], trust_remote_code=True)
print('✅ Processor loaded.')

print(f'\n⏳ Loading model (7B, 4-bit NF4 quantized)...')
print('   First run: ~15 GB download, 10-20 min')
print('   Subsequent runs: ~2-4 min from cache')
model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
    CONFIG['model_id'],
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    attn_implementation='eager',
)
model.eval()

device     = next(model.parameters()).device
vram_used  = torch.cuda.memory_allocated() / 1e9
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'\n✅ Model loaded successfully!')
print(f'   Class          : {type(model).__name__}')
print(f'   Device         : {device}')
print(f'   Quantization   : 4-bit NF4 (double quant enabled)')
print(f'   VRAM used      : {vram_used:.1f} GB / {vram_total:.1f} GB')
print(f'   VRAM available : {vram_total - vram_used:.1f} GB remaining for inference')
print(f'\n⚠️  NOTE: 4-bit models show "torch.uint8" dtype — expected, compute is bfloat16')

GPU before load: 0.00 GB used

⏳ Loading processor: Qwen/Qwen2.5-Omni-7B


preprocessor_config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model-00004-of-00005.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/2.43G [00:00<?, ?B/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

## 🧠 STEP 5: Sentiment Analysis Engine



In [ ]:
# ======================== SentimentAnalyzer ========================
# SentimentAnalyzer -> Evaluates audio segments for emotion and sarcasm using Qwen.
# ||
# ||
# ||
# Functions/Methods -> analyze_call() -> Orchestrates multi-segment processing
# ||                 |
# ||                 |---> __init__() -> Init model & processor state
# ||                 |---> _save_segment_to_temp() -> Array -> .wav temp file
# ||                 |---> _build_messages() -> Construct multimodal prompt
# ||                 |---> _parse_json_response() -> Extract JSON from raw text
# ||                 |---> analyze_segment() -> Pipeline for single audio chunk
# ||                 |---> _aggregate_segments() -> Merge segment data -> call summary
# ||                 |
# ||                 |---> Logic Flow -> Full audio analysis execution:
# ||                                  |
# ||                                  |--- Loop through preprocessed segments
# ||                                  |    └── analyze_segment() -> Inference per chunk
# ||                                  |        ├── _save_segment_to_temp() -> Cache audio
# ||                                  |        ├── _build_messages() -> Format prompt
# ||                                  |        ├── process_mm_info() -> Extract features
# ||                                  |        ├── model.generate() -> Run inference
# ||                                  |        └── _parse_json_response() -> Parse output
# ||                                  |--- _aggregate_segments() -> Compile final summary
# ||
# ======================================================================

# ---------------------------------------------------------------
# SECTION: CLASSES
# ---------------------------------------------------------------
class SentimentAnalyzer:

    EMOTION_LABELS = [
        'happy', 'satisfied', 'neutral', 'confused',
        'not_interested', 'bored', 'irritated',
        'frustrated', 'angry', 'sarcastic',
        'anxious', 'sad', 'excited', 'polite_but_cold'
    ]

    SYSTEM_PROMPT = (
        "You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, "
        "capable of perceiving auditory and visual inputs, as well as generating text and speech."
    )

    ANALYSIS_PROMPT = """You are a forensic audio sentiment analyzer. You will receive a short audio clip.
Always return the JSON output.

STRICT RULES:
1. Listen to the audio COMPLETELY before scoring
2. Every score must reflect what you ACTUALLY heard - not guessed
3. Only ONE emotion should score above 0.6 - the one you are most certain about
4. All other emotions should reflect realistic probabilities
5. NEVER give round numbers like 0.1, 0.2, 0.3 in sequence - that means you are guessing
6. If you are uncertain, set confidence below 0.5 and say so in audio_reasoning

SARCASM DETECTION RULES (critical):
- Sarcasm sounds: overly sweet/calm tone when content should be negative
- Sarcasm sounds: exaggerated politeness, drawn-out words, rising pitch at end of statements
- Sarcasm sounds: mismatch between cheerful tone and tense underlying voice quality
- Sarcasm sounds: emphasis on words like "obviously", "clearly", "sure", "great", "wonderful"
- Sarcasm sounds: voice that is TOO controlled, TOO smooth when talking about a problem
- Sarcasm sounds: slow deliberate pace with slight pitch rise — the speaker is mocking
- Sarcasm sounds: fake agreement — saying "yes" or "absolutely" in a drawn-out flat tone
- BIAS TOWARD TRUE: if there is ANY doubt, mark detected: true with confidence 0.6
- If ANY of these are present → detected: true, confidence > 0.7
- Do NOT mark sarcasm false just because tone sounds "positive" — positive tone ON A PROBLEM = sarcasm
- A calm, sweet voice complaining = almost always sarcastic

WHAT TO LISTEN FOR:
- Tone: Is it genuinely warm or artificially sweet?
- Pace: Natural flow or deliberately slow/exaggerated?
- Pitch: Does it rise sarcastically at end of phrases?
- Stress: Is there tension hiding under a calm surface?
- Hesitations: Pauses, sighs, sharp inhales before speaking?

EMOTION SCORING GUIDE:
- happy: genuine warmth, natural laughter, rising energy
- satisfied: calm, resolved, even tone, natural pace
- neutral: flat, monotone, no emotional coloring
- sarcastic: sweet tone + underlying tension + exaggerated pace
- irritated: clipped words, slightly raised pitch, controlled anger
- frustrated: sighs, repetition stress, strained voice
- angry: loud, fast, sharp consonants, high energy
- anxious: fast pace, higher pitch, voice breaks
- bored: flat, slow, low energy, trailing off
- not_interested: minimal effort, short responses, flat tone
- confused: rising pitch at statements, hesitations, slower pace
- sad: lower pitch, slower pace, reduced energy, voice may break
- excited: fast pace, high pitch variation, high energy
- polite_but_cold: formal tone, clipped politeness, no warmth

OUTPUT FORMAT - return ONLY this JSON, no text before or after:
{
  "primary_emotion": "<exactly one of the 14 labels>",
  "confidence": <genuine 0.0-1.0, NOT rounded>,
  "secondary_emotions": ["<label>", "<label>"],
  "emotion_scores": {
    "happy": <0.0-1.0>, "satisfied": <0.0-1.0>, "neutral": <0.0-1.0>,
    "confused": <0.0-1.0>, "not_interested": <0.0-1.0>, "bored": <0.0-1.0>,
    "irritated": <0.0-1.0>, "frustrated": <0.0-1.0>, "angry": <0.0-1.0>,
    "sarcastic": <0.0-1.0>, "anxious": <0.0-1.0>, "sad": <0.0-1.0>,
    "excited": <0.0-1.0>, "polite_but_cold": <0.0-1.0>
  },
  "vocal_characteristics": {
    "tone": "<warm/cold/tense/neutral/aggressive/artificially_sweet>",
    "pace": "<slow/normal/fast/very_fast/exaggerated>",
    "pitch_variation": "<flat/low/moderate/high/sarcastic_rise>",
    "energy_level": "<low/medium/high>",
    "stress_detected": <true or false>,
    "hesitations_detected": <true or false>,
    "voice_breaks_detected": <true or false>,
    "exaggerated_politeness": <true or false>
  },
  "call_quality_indicators": {
    "engagement_level": "<disengaged/passive/engaged/highly_engaged>",
    "escalation_risk": "<low/medium/high>",
    "resolution_likelihood": "<unlikely/possible/likely>"
  },
  "sarcasm_indicators": {
    "detected": <true or false>,
    "confidence": <0.0-1.0>,
    "signals": ["<specific signal heard>"]
  },
  "overall_sentiment": "<positive/negative/neutral/mixed>",
  "sentiment_score": <-1.0 to 1.0, NOT rounded>,
  "audio_reasoning": "<2-3 sentences: specific vocal cues heard and why they led to this classification>"
}"""

    def __init__(self, model, processor, config):
        self.model     = model
        self.processor = processor
        self.sr        = config['target_sample_rate']
        self.model_id  = config['model_id']

    def _save_segment_to_temp(self, audio_array: "np.ndarray") -> str:
        tmp_path = tempfile.mktemp(suffix='.wav')
        sf.write(tmp_path, audio_array, self.sr)
        return tmp_path

    def _build_messages(self, audio_path: str) -> list:
        return [
            {
                'role': 'system',
                'content': [{'type': 'text', 'text': self.SYSTEM_PROMPT}]
            },
            {
                'role': 'user',
                'content': [
                    {'type': 'audio', 'audio': audio_path},
                    {'type': 'text',  'text': self.ANALYSIS_PROMPT},
                ]
            }
        ]

    def _parse_json_response(self, raw: str) -> dict:
        print(f'\n🔍 RAW MODEL OUTPUT (first 600 chars):\n{raw[:600]}')

        try:
            return json.loads(raw.strip())
        except json.JSONDecodeError:
            pass

        cleaned = re.sub(r'```json\s*|```\s*', '', raw).strip()
        try:
            return json.loads(cleaned)
        except json.JSONDecodeError:
            pass

        matches = list(re.finditer(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', cleaned, re.DOTALL))
        if matches:
            for m in reversed(matches):
                try:
                    return json.loads(m.group())
                except json.JSONDecodeError:
                    continue

        start = cleaned.find('{')
        end   = cleaned.rfind('}')
        if start != -1 and end != -1 and end > start:
            try:
                return json.loads(cleaned[start:end+1])
            except json.JSONDecodeError:
                pass

        attempt = re.sub(r',\s*([}\]])', r'\1', cleaned)
        attempt = re.sub(r'": True',  '": true',  attempt)
        attempt = re.sub(r'": False', '": false', attempt)
        attempt = re.sub(r'": None',  '": null',  attempt)
        start = attempt.find('{')
        end   = attempt.rfind('}')
        if start != -1 and end != -1:
            try:
                return json.loads(attempt[start:end+1])
            except json.JSONDecodeError:
                pass

        print('⚠️  All JSON strategies failed. Salvaging key fields...')
        result = {
            'parse_error'       : True,
            'raw_response'      : raw[:1000],
            'primary_emotion'   : 'unknown',
            'overall_sentiment' : 'unknown',
            'sentiment_score'   : 0.0,
            'confidence'        : 0.0,
            'emotion_scores'    : {},
            'sarcasm_indicators': {'detected': False, 'confidence': 0.0, 'signals': []},
        }
        em   = re.search(r'"primary_emotion"\s*:\s*"([^"]+)"', raw)
        if em:   result['primary_emotion']   = em.group(1)
        sc   = re.search(r'"sentiment_score"\s*:\s*(-?[0-9.]+)', raw)
        if sc:   result['sentiment_score']   = float(sc.group(1))
        ov   = re.search(r'"overall_sentiment"\s*:\s*"([^"]+)"', raw)
        if ov:   result['overall_sentiment'] = ov.group(1)
        conf = re.search(r'"confidence"\s*:\s*([0-9.]+)', raw)
        if conf: result['confidence']        = float(conf.group(1))
        sarc = re.search(r'"detected"\s*:\s*(true|false)', raw)
        if sarc: result['sarcasm_indicators']['detected'] = sarc.group(1) == 'true'
        print(f'   Salvaged: emotion={result["primary_emotion"]}, sentiment={result["overall_sentiment"]}')
        return result

    def analyze_segment(self, audio_array: "np.ndarray",
                        start_sec: float, end_sec: float) -> dict:
        import gc
        gc.collect()
        torch.cuda.empty_cache()

        tmp_audio_path = self._save_segment_to_temp(audio_array)

        try:
            messages   = self._build_messages(tmp_audio_path)
            text_input = self.processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
            )
            audios, images, videos = process_mm_info(messages, use_audio_in_video=False)

            if not audios:
                print(f'   ⚠️  process_mm_info returned empty — loading audio directly')
                audio_fallback, _ = librosa.load(tmp_audio_path, sr=self.sr, mono=True)
                audios = [audio_fallback]
            else:
                print(f'   ✅ Audio loaded by process_mm_info: {len(audios)} clip(s)')

            inputs = self.processor(
                text=text_input, audio=audios, images=images, videos=videos,
                return_tensors='pt', padding=True,
            ).to(next(self.model.parameters()).device)

            with torch.no_grad():
                raw_outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=2048,
                    do_sample=False,
                    repetition_penalty=1.1,
                    return_audio=False,
                )

            output_ids = raw_outputs[0] if isinstance(raw_outputs, tuple) else raw_outputs
            input_len  = inputs['input_ids'].shape[1]
            new_tokens = output_ids[:, input_len:]
            raw_text = self.processor.batch_decode(
                new_tokens, skip_special_tokens=True, clean_up_tokenization_spaces=True,
            )[0]
            sentiment_data = self._parse_json_response(raw_text)

        finally:
            if os.path.exists(tmp_audio_path):
                os.remove(tmp_audio_path)

        sentiment_data['segment_start_sec']    = round(start_sec, 2)
        sentiment_data['segment_end_sec']      = round(end_sec, 2)
        sentiment_data['segment_duration_sec'] = round(end_sec - start_sec, 2)
        return sentiment_data

    def analyze_call(self, preprocessed: dict) -> dict:
        segments = preprocessed['segments']
        print(f'\n🧠 ANALYZING: {Path(preprocessed["original_file"]).name}')
        print(f'   Segments: {len(segments)}')
        print('─' * 50)

        segment_results = []
        for i, (start_sec, end_sec, audio_array) in enumerate(segments):
            print(f'   🔍 Segment {i+1}/{len(segments)} ({start_sec:.1f}s – {end_sec:.1f}s) ... ', end='')
            t0     = time.time()
            result = self.analyze_segment(audio_array, start_sec, end_sec)
            elapsed= time.time() - t0
            print(f"→ {result.get('primary_emotion', 'unknown')} "
                  f"(conf: {result.get('confidence', 0):.2f}) [{elapsed:.1f}s]")
            segment_results.append(result)

        return self._aggregate_segments(segment_results, preprocessed)

    def _aggregate_segments(self, segment_results: list, preprocessed: dict) -> dict:
        valid = [r for r in segment_results if not r.get('parse_error')]

        if not valid:
            return {'error': 'All segments failed to parse', 'segment_details': segment_results}

        def _get_emotion_scores(r):
            return (r.get('emotion_scores') or r.get('emotion_score')
                    or r.get('emotions') or {})

        avg_scores = {}
        for label in self.EMOTION_LABELS:
            scores = [_get_emotion_scores(r).get(label, 0.0) for r in valid]
            avg_scores[label] = round(sum(scores) / len(scores), 3) if scores else 0.0

        dominant_emotion = max(avg_scores, key=avg_scores.get) if avg_scores else 'unknown'

        sentiment_arc = [
            {
                'time'           : f"{r.get('segment_start_sec',0):.1f}s–{r.get('segment_end_sec',0):.1f}s",
                'primary_emotion': r.get('primary_emotion', 'unknown'),
                'sentiment'      : r.get('overall_sentiment', 'unknown'),
                'score'          : r.get('sentiment_score', 0),
            }
            for r in segment_results
        ]

        scores_list = [
            r.get('sentiment_score', 0) for r in valid
            if isinstance(r.get('sentiment_score'), (int, float))
        ]
        avg_sentiment_score = round(sum(scores_list) / len(scores_list), 3) if scores_list else 0.0

        sarcasm_detected = any(
            r.get('sarcasm_indicators', {}).get('detected', False)
            and r.get('sarcasm_indicators', {}).get('confidence', 0) > 0.5
            for r in valid
        )

        risk_map = {'low': 0, 'medium': 1, 'high': 2}

        def _infer_risk(r):
            explicit = r.get('call_quality_indicators', {}).get('escalation_risk')
            if explicit in ('low', 'medium', 'high'):
                return explicit
            scores = (_get_emotion_scores(r))
            hot_score = max(scores.get('angry',0), scores.get('frustrated',0), scores.get('irritated',0))
            if hot_score >= 0.75: return 'high'
            if hot_score >= 0.45: return 'medium'
            return 'low'

        risk_levels  = [_infer_risk(r) for r in valid]
        highest_risk = max(risk_levels, key=lambda x: risk_map.get(x, 0)) if risk_levels else 'low'

        return {
            'metadata': {
                'source_file'          : str(preprocessed['original_file']),
                'analyzed_at'          : datetime.now().isoformat(),
                'model_used'           : self.model_id,
                'call_duration_sec'    : preprocessed['final_duration_sec'],
                'num_segments_analyzed': len(segment_results),
            },
            'call_summary': {
                'dominant_emotion'       : dominant_emotion,
                'overall_sentiment'      : 'positive' if avg_sentiment_score > 0.2
                                           else 'negative' if avg_sentiment_score < -0.2
                                           else 'neutral',
                'sentiment_score'        : avg_sentiment_score,
                'average_emotion_scores' : avg_scores,
                'sarcasm_detected'       : sarcasm_detected,
                'highest_escalation_risk': highest_risk,
                'sentiment_arc'          : sentiment_arc,
            },
            'segment_details': segment_results,
        }

# ---------------------------------------------------------------
# SECTION: MAIN / EXECUTION ENTRY POINT
# ---------------------------------------------------------------
print('✅ SentimentAnalyzer defined.')

## 🔧 STEP 6b: Verify GPU Placement.

In [ ]:


# ── START: GPU placement verification ────────────────────────────────────────

import gc
gc.collect()
torch.cuda.empty_cache()

vram_used  = torch.cuda.memory_allocated() / 1e9
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9

print('🔍 Model placement check:')
param_devices = set()
for name, param in list(model.named_parameters())[:5]:
    print(f'   {name}: {param.device} | dtype: {param.dtype}')
    param_devices.add(str(param.device))

print(f'\nGPU memory: {vram_used:.2f} GB used / {vram_total:.1f} GB total')
print(f'Available : {vram_total - vram_used:.2f} GB free')

if 'cuda:0' in param_devices or any('cuda' in d for d in param_devices):
    print('\n✅ Model is on GPU — ready for inference!')
else:
    print('\n⚠️  Model not on GPU. Forcing move...')
    model = model.to('cuda:0')
    model.eval()
    print(f'   GPU after move: {torch.cuda.memory_allocated()/1e9:.2f} GB used')

print('\n✅ GPU verification complete.')
# ── END: GPU placement verification ──────────────────────────────────────────

In [ ]:


# ── START: emergency GPU fix (run only if inference throws CUDA device errors) ─

import torch, gc
gc.collect()
torch.cuda.empty_cache()
print(f"GPU before: {torch.cuda.memory_allocated()/1e9:.2f} GB used")

for name, param in list(model.named_parameters())[:5]:
    print(f"  {name}: {param.device}")

device = torch.device("cuda:0")
model  = model.to(device)
model.eval()

print(f"\nGPU after: {torch.cuda.memory_allocated()/1e9:.2f} GB used")
print(f"Model device: {next(model.parameters()).device}")
# ── END: emergency GPU fix ────────────────────────────────────────────────────

#BRIDGE FUNCTION


In [ ]:
# ======================== Remote Diarization Client ========================
# Remote Diarization Client -> Fetches speaker segments from a remote API
# ||
# ||
# ||
# Functions/Methods -> get_remote_diarization() -> Main execution flow
# ||                 |
# ||                 |---> CONFIG.get() -> Retrieve diarizer endpoint
# ||                 |---> requests.get() -> Ping bridge health check
# ||                 |---> json() -> Parse health data
# ||                 |---> requests.post() -> Send file path for processing
# ||                 |---> json() -> Parse speaker turns
# ||                 |
# ||                 |---> Logic Flow -> API communication pipeline:
# ||                                  |
# ||                                  |--- IF URL is missing or default
# ||                                  |    └── Return None
# ||                                  |--- requests.get() -> Verify server health
# ||                                  |--- IF server loading/unreachable
# ||                                  |    └── Return None / Warn user
# ||                                  |--- requests.post() -> Send path to /diarize
# ||                                  |--- IF request successful
# ||                                  |    └── Return speaker segments
# ||                                  |--- IF Timeout, HTTPError, or Exception
# ||                                  |    └── Catch -> Print error -> Return None
# ||
# ======================================================================

# ---------------------------------------------------------------
# SECTION: IMPORTS
# ---------------------------------------------------------------
import requests

# ---------------------------------------------------------------
# SECTION: FUNCTIONS
# ---------------------------------------------------------------
def get_remote_diarization(local_pc_audio_path):
    url = CONFIG.get("diarizer_url", "").strip()
    if not url or "YOUR_NGROK_URL_HERE" in url:
        print("❌ diarizer_url is not set in CONFIG.")
        print("   Update CONFIG['diarizer_url'] with your ngrok URL from the terminal.")
        return None

    health_url = url.replace("/diarize", "/health")
    try:
        h = requests.get(health_url, timeout=10)
        if h.status_code == 200:
            hdata = h.json()
            if not hdata.get("pipeline_loaded"):
                print("⏳ Server is up but pipeline still loading — wait 30s and retry.")
                return None
            print(f"✅ Bridge healthy: device={hdata.get('device','?')}")
        else:
            print(f"⚠️  /health returned {h.status_code} — proceeding anyway")
    except requests.exceptions.ConnectionError:
        print(f"❌ Cannot reach server at {health_url}")
        print("   Is server.py running in VS Code? Is ngrok still connected?")
        return None
    except Exception as he:
        print(f"⚠️  /health check failed ({he}) — proceeding anyway")

    payload = {"file_path": local_pc_audio_path}
    try:
        print(f"📡 Sending to diarizer: {local_pc_audio_path}")
        response = requests.post(url, json=payload, timeout=300)
        response.raise_for_status()
        data     = response.json()
        segments = data["segments"]
        print(f"✅ Received {len(segments)} speaker turns | "
              f"{data.get('num_speakers','?')} unique speakers")
        return segments
    except requests.exceptions.HTTPError as e:
        try:
            detail = response.json().get("detail", response.text[:300])
        except Exception:
            detail = response.text[:300]
        print(f"❌ Diarization Bridge HTTP Error: {e}")
        print(f"   Server says: {detail}")
        return None
    except requests.exceptions.Timeout:
        print(f"❌ Request timed out — audio may be very long. Try a shorter clip.")
        return None
    except Exception as e:
        print(f"❌ Diarization Bridge Error: {type(e).__name__}: {e}")
        return None

#STEP 6 : AUDIO UPLOAD & RUN ANALYSIS

In [ ]:
# ======================== Audio Upload & Run Analysis ========================
# Audio Upload & Run Analysis -> Orchestrates upload, diarization, and sentiment pipeline.
# ||
# ||
# ||
# Functions/Methods -> files.upload() -> Trigger file input
# ||                 |
# ||                 |---> AudioPreprocessor() -> Init preprocessor
# ||                 |---> SentimentAnalyzer() -> Init analyzer
# ||                 |---> os.path.exists() -> Check file presence
# ||                 |---> os.listdir() -> Search directory
# ||                 |---> get_remote_diarization() -> Fetch speaker IDs
# ||                 |---> preprocess() -> Format & segment audio
# ||                 |---> librosa.load() -> Load raw audio
# ||                 |---> analyze_segment() -> Inference per chunk
# ||                 |---> _aggregate_segments() -> Merge segment data
# ||                 |---> analyze_call() -> Multi-segment inference
# ||                 |---> os.path.join() -> Build output path
# ||                 |---> json.dump() -> Save results to JSON
# ||                 |---> traceback.print_exc() -> Output error logs
# ||                 |
# ||                 |---> Logic Flow -> End-to-end execution pipeline:
# ||                                  |
# ||                                  |--- files.upload() -> Get user files
# ||                                  |--- Loop through uploaded_files
# ||                                  |    ├── IF os.path.exists() is False
# ||                                  |    │   └── os.listdir() -> Search or skip
# ||                                  |    ├── get_remote_diarization() -> Fetch metadata
# ||                                  |    ├── preprocess() -> Format audio
# ||                                  |    ├── IF metadata exists
# ||                                  |    │   ├── librosa.load() -> Load full audio
# ||                                  |    │   ├── analyze_segment() -> Analyze full duration
# ||                                  |    │   └── _aggregate_segments() -> Merge with metadata
# ||                                  |    └── ELSE
# ||                                  |        └── analyze_call() -> Use default chunks
# ||                                  |    ├── os.path.join() -> Prepare file path
# ||                                  |    └── json.dump() -> Save output
# ||                                  |--- IF Exception occurs
# ||                                  |    └── traceback.print_exc() -> Log error
# ||
# ======================================================================

# ---------------------------------------------------------------
# SECTION: IMPORTS
# ---------------------------------------------------------------
from google.colab import files
from pathlib import Path
import os, json, time, traceback

# ---------------------------------------------------------------
# SECTION: MAIN / EXECUTION ENTRY POINT
# ---------------------------------------------------------------
print('📁 Click "Choose Files" to upload your audio file(s):')
uploaded       = files.upload()
uploaded_files = list(uploaded.keys())
print(f'\n✅ Uploaded {len(uploaded_files)} file(s):')
for f in uploaded_files:
    print(f'   📎 {f} ({len(uploaded[f])/1024:.1f} KB)')

assert 'model'     in dir(), '❌ model not defined — run Step 4 first!'
assert 'processor' in dir(), '❌ processor not defined — run Step 4 first!'

preprocessor = AudioPreprocessor(CONFIG)
analyzer     = SentimentAnalyzer(model, processor, CONFIG)
all_results  = {}

for audio_filename in uploaded_files:
    print(f'\n{"="*60}')
    print(f'📞 Processing: {audio_filename}')
    print('='*60)

    audio_path = f'/content/{audio_filename}'

    if not os.path.exists(audio_path):
        print(f'⚠️  Not found at {audio_path}, searching...')
        matches = [f for f in os.listdir('/content') if audio_filename in f]
        if matches:
            audio_path = f'/content/{matches[0]}'
            print(f'   Found at: {audio_path}')
        else:
            print(f'❌ Could not locate file. Skipping.')
            continue

    try:
        pc_file_path = f"C:\\Users\\Pcc\\Downloads\\diarization\\{audio_filename}"
        speaker_metadata = get_remote_diarization(pc_file_path)

        preprocessed = preprocessor.preprocess(audio_path)

        if speaker_metadata:
            print(f"🎙️ Running Full-Audio Analysis with Speaker Tags...")

            full_audio, _ = librosa.load(
                audio_path, sr=CONFIG['target_sample_rate'], mono=True
            )
            full_duration = len(full_audio) / CONFIG['target_sample_rate']
            print(f"   Full audio: {full_duration:.1f}s — sending as single segment")
            print(f"   Speaker timeline: {len(speaker_metadata)} turns from diarization")

            result = analyzer.analyze_segment(
                full_audio, 0.0, full_duration
            )

            if result.get('parse_error') or result.get('primary_emotion') == 'unknown':
                print(f"   ⚠️  Refusal on first try — retrying full audio...")
                result = analyzer.analyze_segment(
                    full_audio, 0.0, full_duration
                )

            result['speaker_timeline'] = speaker_metadata
            result['speaker_summary']  = {
                spk: round(sum(
                    s['end'] - s['start']
                    for s in speaker_metadata if s['speaker'] == spk
                ), 2)
                for spk in set(s['speaker'] for s in speaker_metadata)
            }
            print(f"   Speaker summary: {result['speaker_summary']}")

            segment_results = [result]
            results = analyzer._aggregate_segments(segment_results, preprocessed)

        else:
            print("⚠️ Bridge failed. Falling back to default segmentation.")
            results = analyzer.analyze_call(preprocessed)

        stem             = Path(audio_filename).stem
        output_json_path = os.path.join(CONFIG["output_dir"], f"{stem}_sentiment.json")
        with open(output_json_path, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

        all_results[audio_filename] = results

        summary = results.get('call_summary', {})
        print(f'\n{"─"*50}')
        print(f'📊 RESULTS: {audio_filename}')
        print(f'   🎭 Dominant emotion  : {summary.get("dominant_emotion", "N/A")}')
        print(f'   💬 Overall sentiment : {summary.get("overall_sentiment", "N/A")} '
              f'(score: {summary.get("sentiment_score", 0):.3f})')
        print(f'   🎯 Sarcasm detected  : {summary.get("sarcasm_detected", False)}')
        print(f'   ⚠️  Escalation risk   : {summary.get("highest_escalation_risk", "N/A")}')
        print(f'   💾 Saved to          : {output_json_path}')

    except Exception as e:
        print(f'\n❌ Failed: {e}')
        traceback.print_exc()
        all_results[audio_filename] = {'error': str(e), 'file': audio_filename}

print(f'\n✅ Done! Processed {len(all_results)}/{len(uploaded_files)} file(s).')

## 📤 STEP 7: View & Download Results

In [ ]:
# ======================== Display Results ========================
# Display Results -> Retrieves and prints the final JSON output of the analysis.
# ||
# ||
# ||
# Functions/Methods -> all_results.get() -> Fetch analysis from memory
# ||                 |
# ||                 |---> json.dumps() -> Format dictionary to JSON string
# ||                 |---> print() -> Display output to console
# ||                 |
# ||                 |---> Logic Flow -> Display pipeline:
# ||                                  |
# ||                                  |--- IF all_results is empty
# ||                                  |    └── print() -> Show missing results error
# ||                                  |--- ELSE
# ||                                  |    ├── Get first filename from uploaded_files
# ||                                  |    ├── all_results.get() -> Fetch result dictionary
# ||                                  |    ├── IF result is None
# ||                                  |    │   └── print() -> Show not found error
# ||                                  |    ├── ELIF 'error' in result
# ||                                  |    │   └── print() -> Show processing failure error
# ||                                  |    └── ELSE
# ||                                  |        └── json.dumps() -> Pretty-print and display JSON
# ||
# ======================================================================

# ---------------------------------------------------------------
# SECTION: MAIN / EXECUTION ENTRY POINT
# ---------------------------------------------------------------
if not all_results:
    print('❌ No results — run Step 6 first.')
else:
    filename_to_show = uploaded_files[0]
    result = all_results.get(filename_to_show)
    if result is None:
        print(f'❌ No result found for {filename_to_show}')
    elif 'error' in result:
        print(f'❌ Processing failed: {result["error"]}')
    else:
        print(f'📋 FULL RESULT: {filename_to_show}')
        print('═' * 60)
        print(json.dumps(result, indent=2))

In [ ]:
# ======================== Download Results ========================
# Download Results -> Triggers browser downloads for generated JSON reports.
# ||
# ||
# ||
# Functions/Methods -> Path().stem -> Extract filename without extension
# ||                 |
# ||                 |---> os.path.join() -> Build output file path
# ||                 |---> os.path.exists() -> Verify file existence
# ||                 |---> colab_files.download() -> Trigger browser download
# ||                 |---> print() -> Log download status
# ||                 |
# ||                 |---> Logic Flow -> Batch download pipeline:
# ||                                  |
# ||                                  |--- Loop through uploaded_files
# ||                                  |    ├── Path().stem & os.path.join() -> Build path
# ||                                  |    ├── IF os.path.exists() is True
# ||                                  |    │   ├── colab_files.download() -> Save to local
# ||                                  |    │   └── print() -> Log success
# ||                                  |    └── ELSE
# ||                                  |        └── print() -> Warn not found
# ||
# ======================================================================

# ---------------------------------------------------------------
# SECTION: IMPORTS
# ---------------------------------------------------------------
from google.colab import files as colab_files

# ---------------------------------------------------------------
# SECTION: MAIN / EXECUTION ENTRY POINT
# ---------------------------------------------------------------
print('📥 Downloading result JSON files...')
for audio_filename in uploaded_files:
    stem      = Path(audio_filename).stem
    json_path = os.path.join(CONFIG['output_dir'], f'{stem}_sentiment.json')
    if os.path.exists(json_path):
        colab_files.download(json_path)
        print(f'  ✅ Downloaded: {stem}_sentiment.json')
    else:
        print(f'  ❌ Not found: {json_path}')

In [ ]:
# ======================== Sentiment Visualization Dashboard ========================
# Sentiment Visualization Dashboard -> Generates 2-panel emotion and sentiment arc chart
# ||
# ||
# ||
# Functions/Methods -> %matplotlib inline -> Set plot backend
# ||                 |
# ||                 |---> plt.figure() -> Init canvas
# ||                 |---> gridspec.GridSpec() -> Define vertical layout
# ||                 |---> ax1.barh() -> Plot horizontal bars
# ||                 |---> ax2.fill_between() -> Shade polarity zones
# ||                 |---> ax2.plot() -> Draw timeline arc
# ||                 |---> fig.savefig() -> Save to PNG
# ||                 |---> plt.show() -> Render in notebook
# ||                 |
# ||                 |---> Logic Flow -> Plotting execution:
# ||                                  |
# ||                                  |--- IF no results -> Show error
# ||                                  |--- ELSE
# ||                                  |    ├── Find valid result with 'call_summary'
# ||                                  |    ├── IF no valid result -> Show error
# ||                                  |    └── ELSE
# ||                                  |        ├── Extract summary, arc, scores
# ||                                  |        ├── Init canvas & GridSpec
# ||                                  |        ├── Define styling & colors
# ||                                  |        ├── ax1.barh() -> Render bar chart
# ||                                  |        ├── IF arc data exists
# ||                                  |        │   ├── ax2.fill_between() -> Color zones
# ||                                  |        │   └── ax2.plot() & ax2.scatter() -> Plot timeline
# ||                                  |        ├── Apply axis styling
# ||                                  |        ├── fig.savefig() -> Export PNG
# ||                                  |        └── plt.show() -> Display
# ||
# ======================================================================

# ---------------------------------------------------------------
# SECTION: IMPORTS
# ---------------------------------------------------------------
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi'] = 150  # Increased DPI for crisper charts
import matplotlib.gridspec as gridspec
import numpy as np
from pathlib import Path

# ---------------------------------------------------------------
# SECTION: MAIN / EXECUTION ENTRY POINT
# ---------------------------------------------------------------
if not all_results:
    print("❌  No results found — run Step 6 first!")
else:
    result, chosen_file = None, None
    for fname, res in all_results.items():
        if "error" not in res and "call_summary" in res:
            result, chosen_file = res, fname
            break

    if result is None:
        print("❌  No valid results to visualise.")
    else:
        summary  = result["call_summary"]
        segments = result.get("segment_details", [])
        meta     = result.get("metadata", {})

        EMOTION_LABELS = [
            "happy", "satisfied", "neutral", "confused",
            "not_interested", "bored", "irritated",
            "frustrated", "angry", "sarcastic",
            "anxious", "sad", "excited", "polite_but_cold",
        ]

        # Enhanced color palette for better contrast and modern look
        EMOTION_COLORS = {
            "happy": "#FFD700", "satisfied": "#98FB98", "neutral": "#B0C4DE",
            "confused": "#DDA0DD", "not_interested": "#808080", "bored": "#A9A9A9",
            "irritated": "#FFA07A", "frustrated": "#FF8C00", "angry": "#FF4500",
            "sarcastic": "#BA55D3", "anxious": "#FF69B4", "sad": "#4169E1",
            "excited": "#00FFFF", "polite_but_cold": "#708090",
        }

        avg_scores = summary.get("average_emotion_scores", {})
        scores_arr = np.array([avg_scores.get(e, 0.0) for e in EMOTION_LABELS])

        arc      = summary.get("sentiment_arc", [])
        arc_x    = list(range(1, len(arc) + 1))
        arc_vals = [s.get("score", 0) for s in arc]
        arc_emos = [s.get("primary_emotion", "neutral") for s in arc]
        arc_time = [s.get("time", f"Seg {i+1}") for i, s in enumerate(arc)]

        # Refined Dark Mode theme
        DARK_BG, PANEL_BG = "#121212", "#1E1E1E"
        TEXT_COL, GRID_COL, ACCENT = "#E0E0E0", "#333333", "#BB86FC"

        # Taller figure for better spacing
        fig = plt.figure(figsize=(12, 16), facecolor=DARK_BG)
        gs  = gridspec.GridSpec(2, 1, figure=fig, hspace=0.35, top=0.92, bottom=0.08, left=0.15, right=0.95)

        ax1 = fig.add_subplot(gs[0, 0])
        ax2 = fig.add_subplot(gs[1, 0])

        def style_ax(ax, title):
            ax.set_facecolor(PANEL_BG)
            for sp in ax.spines.values():
                sp.set_color(GRID_COL)
                sp.set_linewidth(1.5) # Thicker borders
            ax.tick_params(colors=TEXT_COL, labelsize=10, length=6, width=1.5)
            ax.set_title(title, color=TEXT_COL, fontsize=14, fontweight="bold", pad=15)
            ax.yaxis.label.set_color(TEXT_COL)
            ax.yaxis.label.set_fontsize(12)
            ax.xaxis.label.set_color(TEXT_COL)
            ax.xaxis.label.set_fontsize(12)

        bar_colors = [EMOTION_COLORS[e] for e in EMOTION_LABELS]
        # Added slight rounded effect by removing edge colors and adjusting width
        bars = ax1.barh(EMOTION_LABELS, scores_arr, color=bar_colors, edgecolor="none", height=0.7)
        for bar, val in zip(bars, scores_arr):
            if val > 0.01: # Lowered threshold to show more values
                ax1.text(val + 0.015, bar.get_y() + bar.get_height() / 2, f"{val:.2f}",
                         va="center", ha="left", color=TEXT_COL, fontsize=9, fontweight='medium')

        ax1.set_xlim(0, max(scores_arr) + 0.15 if len(scores_arr) > 0 and max(scores_arr) > 0 else 1.1)
        ax1.set_xlabel("Confidence Score (0 – 1)", color=TEXT_COL)
        style_ax(ax1, "🎭  Emotion Score Distribution")
        ax1.invert_yaxis()
        ax1.xaxis.grid(True, color=GRID_COL, linestyle='--', linewidth=0.8, alpha=0.7)
        ax1.set_axisbelow(True) # Put grid behind bars

        if arc_vals:
            arc_colors = [EMOTION_COLORS.get(e, ACCENT) for e in arc_emos]

            # Smooth gradient-like fills
            ax2.fill_between(arc_x, arc_vals, 0, where=[v >= 0 for v in arc_vals],
                             alpha=0.25, color="#4CAF50", interpolate=True)
            ax2.fill_between(arc_x, arc_vals, 0, where=[v < 0 for v in arc_vals],
                             alpha=0.25, color="#F44336", interpolate=True)

            # Thicker line with markers
            ax2.plot(arc_x, arc_vals, color=ACCENT, linewidth=2.5, zorder=4, alpha=0.8)
            ax2.scatter(arc_x, arc_vals, c=arc_colors, s=100, zorder=5,
                        edgecolors=DARK_BG, linewidths=1.5) # Dark edge for pop

            ax2.axhline(0, color="#666666", linewidth=1.5, linestyle="--", zorder=3)
            ax2.set_ylim(-1.1, 1.1)
            ax2.set_xticks(arc_x)
            ax2.set_xticklabels(arc_time, rotation=35, ha="right", fontsize=9)
            ax2.set_ylabel("Sentiment Polarity (-1 to 1)")
            ax2.yaxis.grid(True, color=GRID_COL, linestyle='--', linewidth=0.8, alpha=0.7)
            ax2.set_axisbelow(True)

        style_ax(ax2, "📈  Sentiment Arc Timeline")

        fig.suptitle(f"Analysis Dashboard | {Path(chosen_file).name}",
                     color=TEXT_COL, fontsize=16, fontweight="bold", y=0.96)

        out_path = f"{CONFIG['output_dir']}/{Path(chosen_file).stem}_dashboard.png"
        fig.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=DARK_BG)
        plt.show()

        print(f"\n✅ Dashboard saved → {out_path}")